In [ ]:
# General library imports.
import sys
from pathlib import Path

# Find the example helpers from either the repository root or this directory.
notebook_dir = Path.cwd() if (Path.cwd() / "text_abstention.py").exists() else Path.cwd() / "quickstart_ipynbs"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

%load_ext autoreload
%autoreload 2
%aimport -torch -sklearn -numpy -np -pandas -tensorflow -transformers -trl

from itertools import islice
from datasets import load_dataset
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Install first if needed: %pip install datasets pillow pandas matplotlib
REPO_ID = "cvl-umass/RealBirdID"
REVISION = "abd12f9a663f47bc64b8345eeb54679d134762aa"
SPLIT = "validation"  # Both configs are validation-only, not train/test.
MAX_ROWS = 250  # Per config. Set None to count the full release (more I/O).
SEED = 42
columns = ["observation_id", "taxon_name", "taxon_rank", "reasons", "license", "source_license_code"]

In [ ]:
# Read metadata only; do not decode images to count labels and licenses.
metadata = []
for subset in ("answerable", "unanswerable"):
    stream = load_dataset(REPO_ID, subset, split=SPLIT, revision=REVISION, streaming=True)
    stream = stream.select_columns(columns)
    for row in islice(stream, MAX_ROWS):
        metadata.append({"subset": subset, "split": SPLIT, **row})
metadata = pd.DataFrame(metadata)
assert not metadata.empty
scope = "Full release" if MAX_ROWS is None else f"First {MAX_ROWS} rows per config (not a random sample)"
print(scope)
display(metadata.groupby(["subset", "split", "taxon_rank"]).size().rename("rows").reset_index())

In [ ]:
# Answerable labels are species; unanswerable labels can be higher-rank taxa.
label_counts = metadata.groupby(["subset", "taxon_rank", "taxon_name"]).size().rename("rows")
license_counts = pd.crosstab(metadata["license"], metadata["subset"])
display(label_counts.sort_values(ascending=False).head(15).reset_index())
display(license_counts)
print("Null source license codes:", int(metadata["source_license_code"].isna().sum()))
# 'unknown' and 'all-rights-reserved' are distinct from a CC license.

In [ ]:
# Use the optional mu figure wrapper when installed; Matplotlib is sufficient otherwise.
import importlib.util
if importlib.util.find_spec("mu") is not None:
    from mu.visv2.figure import Figure
    chart = Figure(n_rows=1, n_cols=3, figsize=(16, 5), fontfamily="serif")
    fig = chart.fig
    axes = [panel.p for panel in chart.panels]
else:
    plt.rcParams["font.family"] = "serif"
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, subset in zip(axes[:2], ("answerable", "unanswerable")):
    counts = metadata.loc[metadata["subset"] == subset, "taxon_name"].value_counts().head(10)
    counts.sort_values().plot.barh(ax=ax, title=f"{subset}: top 10 labels")
    ax.set_xlabel("Rows")
license_counts.plot.barh(ax=axes[2], title="Photo licenses")
axes[2].set_xlabel("Rows")
fig.suptitle(scope)
fig.tight_layout()
if importlib.util.find_spec("mu") is not None:
    chart.show()
else:
    plt.show()

In [ ]:
# Browse by config/split and taxon. For answerable rows this is a species filter.
SUBSET = "answerable"  # Try "unanswerable" to browse genus-level labels.
TAXON_NAME = metadata.loc[metadata["subset"] == SUBSET, "taxon_name"].iloc[0]
N_IMAGES = 3
SCAN_LIMIT = 1000  # Bound the scan when the selected label is rare.

stream = load_dataset(REPO_ID, SUBSET, split=SPLIT, revision=REVISION, streaming=True)
# Keep images encoded until a matching row is selected.
from datasets import Image
from io import BytesIO
from PIL import Image as PILImage
stream = stream.cast_column("image", Image(decode=False))
matches = []
for row in islice(stream, SCAN_LIMIT):
    if row["taxon_name"] == TAXON_NAME:
        matches.append(row)
        if len(matches) == N_IMAGES:
            break
print(f"{len(matches)} matches for {TAXON_NAME!r} in the first {SCAN_LIMIT} rows of {SUBSET}/{SPLIT}")
for row in matches:
    print(row["taxon_name"], row["taxon_rank"], row["reasons"])
    print("License:", row["license"], row["license_url"])
    print("Credit:", row["photo_attribution"])
    print("Source:", row["source_url"])
    encoded = row["image"]
    source = BytesIO(encoded["bytes"]) if encoded["bytes"] is not None else encoded["path"]
    with PILImage.open(source) as image:
        display(image.copy())